# Post-training
后训练微调

pretrain -> post train

把只会续写的基座模型，变成能听懂指令的对话模型

# SFT

lm_dataset：数据增强

`pre_processing_chat`：对话前预处理，因为我们希望sft学会对话

``

In [ ]:
def pre_processing_chat(conversations, add_system_ratio=0.2):
    """对对话数据做轻量预处理。

    - 如果样本里包含工具调用相关字段，则直接返回，避免破坏 tool-use 结构。
    - 普通对话按概率补一个 system 角色，增强模型对系统提示词的鲁棒性。
    """
    # tool use 数据完整保留不做处理
    if any(conv.get('tools') for conv in conversations): return conversations

    SYSTEM_PROMPTS = [
        "你是一个知识丰富的AI，尽力为用户提供准确的信息。",
        "你是minimind，一个小巧但有用的语言模型。",
        "你是一个专业的AI助手，请提供有价值的回答。",
        "你是minimind，请尽力帮助用户解决问题。",
        "你是一个可靠的AI，请给出准确的回答。",
        "You are a helpful AI assistant.",
        "You are minimind, a lightweight intelligent assistant.",
        "You are a friendly chatbot. Please answer the user's questions carefully.",
        "You are a knowledgeable AI. Try your best to provide accurate information.",
        "You are minimind, a small but useful language model."
    ]
    # 概率性添加system
    if conversations[0].get('role') != 'system':
        if random.random() < add_system_ratio:
            return [{'role': 'system', 'content': random.choice(SYSTEM_PROMPTS)}] + conversations
    return conversations

def post_processing_chat(prompt_content, empty_think_ratio=0.2):
    """对拼好的 prompt 做后处理。"""
    # 以 80% 的概率移除空思考标签，减少训练时出现无意义的 <think></think>
    if '<think>\n\n</think>\n\n' in prompt_content and random.random() > empty_think_ratio:
        prompt_content = prompt_content.replace('<think>\n\n</think>\n\n', '')
    return prompt_content

## 1.2 `SFTDataset` (有监督微调数据集)

监督微调数据集，把多轮对话拼成训练 prompt，只让 assistant 回复部分参与 loss。
- **用途**：用于指令微调（Instruction Tuning）。
- **核心难点**：**Loss Masking（损失掩码）**。在对话中，我们只希望模型学习“助手”的回答，而不学习“用户”的提问。
- **实现细节**：
  - `create_chat_prompt`：应用对话模板。
  - `generate_labels`：这是最关键的函数。它初始化全为 `-100`（忽略 Loss）的标签列表。然后扫描 Input IDs，找到“助手”回答的片段（介于 `bos_token`+`assistant` 和 `eos_token` 之间），将这部分的 Label 设为真实的 Token ID。
  - **效果**：模型只针对 Assistant 的输出计算梯度。

这是用于SFT的数据集处理，

In [ ]:
class SFTDataset(Dataset):
    """监督微调数据集（SFT）。

    作用：
    - 读取多轮对话数据
    - 按 tokenizer 的 chat template 拼成最终 prompt
    - 只让 assistant 的回答部分参与监督

    也就是说，这个数据集的核心不是“预测整段对话”，而是
    “给定上下文，学习生成 assistant 回复”。
    """
    def __init__(self, jsonl_path, tokenizer, max_length=1024):
        super().__init__()
        self.tokenizer = tokenizer
        self.max_length = max_length

        # 显式定义数据结构，帮助 datasets 库正确解析 JSON 中的嵌套字段。
        # conversations 是一个 message 列表，每条 message 允许包含这些字段。
        # 这里保留 reasoning_content、tools、tool_calls，是为了兼容推理/工具调用格式。
        features = Features({
            'conversations': [{
                'role': Value('string'),
                'content': Value('string'),
                'reasoning_content': Value('string'),
                'tools': Value('string'),
                'tool_calls': Value('string')
            }]
        })
        self.samples = load_dataset('json', data_files=jsonl_path, split='train', features=features)

        # 这两个 token 序列用于在最终 token 串里定位 assistant 回复区间。
        self.bos_id = tokenizer(f'{tokenizer.bos_token}assistant\n', add_special_tokens=False).input_ids
        # BOS：标注 assistant 回复开始的标记；EOS：assistant 回复结束的标记。
        self.eos_id = tokenizer(f'{tokenizer.eos_token}\n', add_special_tokens=False).input_ids

    def __len__(self):
        return len(self.samples)

    def create_chat_prompt(self, conversations):
        """把原始对话列表转换成最终训练 prompt。

        这里会做两件额外处理：
        1. 如果 system 消息里带了 tools，把 tools 取出来单独传给 template。
        2. 如果 tool_calls 是字符串，先转成真正的 Python 对象。
        """
        # 复制一份，避免在原始样本对象上直接修改。
        messages = []
        tools = None

        for message in conversations:
            message = dict(message)

            # 工具定义通常放在 system 消息里。
            # 这里把 tools 单独抽出来，后面作为 apply_chat_template 的参数传入。
            if message.get("role") == "system" and message.get("tools"):
                tools = json.loads(message["tools"]) if isinstance(message["tools"], str) else message["tools"]

            # tool_calls 如果是字符串形式，先反序列化成对象。
            if message.get("tool_calls") and isinstance(message["tool_calls"], str):
                message["tool_calls"] = json.loads(message["tool_calls"])

            messages.append(message)

        # 交给 tokenizer 的 chat template 统一拼接。
        # add_generation_prompt=False 的含义是：这里生成的是完整训练文本，
        # 不是留一个空的 assistant 起始位等模型接着生成。
        return self.tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=False,
            tools=tools
        )

    def generate_labels(self, input_ids):
        """根据 token 序列生成监督标签。

        逻辑：
        - 默认所有位置都设为 -100，表示不参与 loss
        - 扫描整个 token 序列，找到 assistant 回复段
        - 将 assistant 回复段内的 token 标记为真实标签

        这样做的好处是：
        - user/system prompt 不参与 loss
        - 模型只学习“该回答什么”
        """
        labels = [-100] * len(input_ids)
        i = 0

        while i < len(input_ids):
            # 找 assistant 回复段的起始标记。
            if input_ids[i:i + len(self.bos_id)] == self.bos_id:
                start = i + len(self.bos_id)
                end = start

                # 从 start 开始往后找 EOS，定位这一段 assistant 回复的结束位置。
                while end < len(input_ids):
                    if input_ids[end:end + len(self.eos_id)] == self.eos_id:
                        break
                    end += 1

                # assistant 回复段内的 token 全部作为监督目标。
                # 这里把 EOS 也算进去，帮助模型学习“如何正确结束回答”。
                for j in range(start, min(end + len(self.eos_id), self.max_length)):
                    labels[j] = input_ids[j]

                # 跳到这一段结束后的下一个位置，继续寻找后续 assistant 回复。
                i = end + len(self.eos_id) if end < len(input_ids) else len(input_ids)
            else:
                i += 1

        return labels

    def __getitem__(self, index):
        sample = self.samples[index]
        # 1. 预处理对话，按概率随机选择是否要插入system_prompt
        conversations = pre_processing_chat(sample['conversations'])

        # 2. 拼成模板化 prompt，用create_chat_prompt把对话转化成文本
        prompt = self.create_chat_prompt(conversations)

        # 3. 做后处理，比如移除空 think 标签
        prompt = post_processing_chat(prompt)

        # 4. 对token序列进行截断--max_length是可处理的最大序列长度，tokenization
        input_ids = self.tokenizer(prompt).input_ids[:self.max_length]
        # 5. 补足 padding 到固定长度，便于 batch 训练
        input_ids += [self.tokenizer.pad_token_id] * (self.max_length - len(input_ids))

        # 6. 生成只监督 assistant 的 labels
        labels = self.generate_labels(input_ids)

        # 下面这段是调试打印，默认注释掉。
        # 如果你想检查 label 对齐是否正确，可以打开。
        # print(f"\n--- Sample {index} ---")
        # for i, (x, y) in enumerate(zip(input_ids[:-1], labels[1:])):
        #     print(f"{i:3d}: X={self.tokenizer.decode([x])!r:16s} ---> Y={self.tokenizer.decode([input_ids[i+1]])!r:16s} label={y}")

        return torch.tensor(input_ids, dtype=torch.long), torch.tensor(labels, dtype=torch.long)

# LoRA

In [ ]:

# 定义Lora网络结构
class LoRA(nn.Module):
    def __init__(self, in_features, out_features, rank):
        super().__init__()
        self.rank = rank  # LoRA的秩（rank），控制低秩矩阵的大小
        self.A = nn.Linear(in_features, rank, bias=False)  # 低秩矩阵A
        self.B = nn.Linear(rank, out_features, bias=False)  # 低秩矩阵B
        # 矩阵A高斯初始化
        self.A.weight.data.normal_(mean=0.0, std=0.02)
        # 矩阵B全0初始化
        self.B.weight.data.zero_()

    def forward(self, x):
        return self.B(self.A(x))


def apply_lora(model, rank=16):
    for name, module in model.named_modules():
        if isinstance(module, nn.Linear) and module.in_features == module.out_features:
            lora = LoRA(module.in_features, module.out_features, rank=rank).to(model.device)
            # 创建一个lora对象，并将其绑定到原始模块上
            setattr(module, "lora", lora)
            original_forward = module.forward

            # 显式绑定
            def forward_with_lora(x, layer1=original_forward, layer2=lora):
                return layer1(x) + layer2(x)
                # lora输出 + 原始输出

            module.forward = forward_with_lora

train

构建lora模型，并接入原模型。并冻结非lora参数（即原模型参数）从而只对lora矩阵A，B进行训练

In [ ]:
    # ========== 5. 定义模型、应用LoRA、冻结非LoRA参数 ==========
    model, tokenizer = init_model(lm_config, args.from_weight, device=args.device)
    apply_lora(model)
    
    # 统计参数
    total_params = sum(p.numel() for p in model.parameters())
    lora_params_count = sum(p.numel() for name, p in model.named_parameters() if 'lora' in name)
    Logger(f"LLM 总参数量: {total_params / 1e6:.3f} M")
    Logger(f"LoRA 参数量: {lora_params_count / 1e6:.3f} M")
    Logger(f"LoRA 参数占比: {lora_params_count / total_params * 100:.2f}%")
    
    # 冻结非LoRA参数，收集LoRA参数
    lora_params = []
    for name, param in model.named_parameters():
        if 'lora' in name:
            param.requires_grad = True
            lora_params.append(param)
        else:
            param.requires_grad = False